In [0]:
from pyspark.sql.functions import monotonically_increasing_id, lit

table_name = "workspace.ecommerce.user_features_silver"
target_df = spark.table(table_name)

spark.sql(f"DESCRIBE HISTORY {table_name}").select("version", "timestamp", "operation").show(3, truncate=False)

print("\nAppending New Fake Records")

fake_df = target_df.limit(2) \
    .withColumn("user_id", (monotonically_increasing_id() + 999990000).cast(target_df.schema["user_id"].dataType)) \
    .withColumn("total_purchases", lit(999).cast(target_df.schema["total_purchases"].dataType))

fake_df.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
latest_version = history_df.select("version").head()[0]

previous_version = latest_version - 1 

print(f"Current Corrupted Version: {latest_version}")
print(f"Previous Clean Version: {previous_version}")

clean_old_df = spark.read.option("versionAsOf", previous_version).table(table_name)
corrupted_new_df = spark.read.table(table_name)

print(f"\nRow Count in Old Version: {clean_old_df.count()}")
print(f"Row Count in New Version: {corrupted_new_df.count()}")

In [0]:
# Compare differences
differences_df = corrupted_new_df.exceptAll(clean_old_df)

display(differences_df)

# Restoring the table to fix the mistake permanently
spark.sql(f"RESTORE TABLE {table_name} TO VERSION AS OF {previous_version}")

print("Table successfully restored to its original clean state")